# Polars Lab: локальная учебная реализация того же инкрементального хранилища

Этот notebook повторяет тот же контракт, что и Spark SQL-версия, но полностью работает локально в Polars.
Он предназначен для учебного прогона, быстрой отладки и понятного воспроизведения логики без отдельного Spark-кластера.

In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import re
import shutil
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Any

import polars as pl

PARTITION_NAME_PATTERN = re.compile(r"^(?P<key>[A-Za-z0-9_]+)='(?P<value>.*)'$")
FUTURE_DTTM = dt.datetime(9999, 12, 31, 0, 0, 0)

SOURCE_REGISTRY = {
    'client_cards_daily': {
        'source_id': 1,
        'source_description': 'Daily client card source.',
        'update_frequency': 'daily',
        'target_table': 'client_daily_attrs_scd2',
        'partition_key': 'row_actual_to',
        'columns': (
            'srv_mb_nflag',
            'cc_stoplist',
            'lne_tot_debt_int_ovrd_rub_amt',
            'lne_tot_debt_ovrd_rub_amt',
        ),
    },
    'credit_cards_info': {
        'source_id': 2,
        'source_description': 'Monthly credit card source.',
        'update_frequency': 'monthly',
        'target_table': 'client_monthly_attrs_scd1',
        'partition_key': 'report_dt',
        'columns': (
            'client_income_amt',
            'oi_total_amt',
            'act_pl_os_rub_amt',
            'payroll_client_nflag',
            'inf_payroll_rub_amt',
            'legal_entity_amt',
            'inc_avg_risk_rub_amt',
            'otf_loan_rub_amt',
            'otf_fee_rub_amt',
            'inf_transfer_rub_amt',
            'cc_ever_nflag',
        ),
    },
    'deb_cards_info': {
        'source_id': 3,
        'source_description': 'Monthly debit card source.',
        'update_frequency': 'monthly',
        'target_table': 'client_monthly_attrs_scd1',
        'partition_key': 'report_dt',
        'columns': (
            'onl_bank_active_1m_nfalg',
            'auto_pay_active_qty',
            'cl_income_1m_amt',
            'dep_acc_1st_open_dt',
            'wdr_cash_6m_amt',
            'cash_op_6m_amt',
            'cash_3m_qty',
            'lst_balance_amt',
            'card_active_1m_nflag',
        ),
    },
}

ATTRIBUTE_ID_BY_NAME = {
    attribute_name: attribute_id
    for attribute_id, attribute_name in enumerate(
        [
            column_name
            for source_meta in SOURCE_REGISTRY.values()
            for column_name in source_meta['columns']
        ],
        start=1,
    )
}

POLARS_TABLE_SCHEMAS = {
    'dim_sources': {
        'source_id': pl.Int32,
        'source_name': pl.String,
        'source_description': pl.String,
        'update_frequency': pl.String,
        'row_create_dtime': pl.Datetime,
        'row_update_dtime': pl.Datetime,
        'valid_from': pl.Datetime,
        'valid_to': pl.Datetime,
    },
    'dim_attributes': {
        'attribute_id': pl.Int32,
        'attribute_name': pl.String,
        'attribute_description': pl.String,
        'data_type': pl.String,
        'source_id': pl.Int32,
        'update_frequency': pl.String,
        'row_create_dtime': pl.Datetime,
        'row_update_dtime': pl.Datetime,
    },
    'load_log': {
        'load_id': pl.Int64,
        'source_id': pl.Int32,
        'source_name': pl.String,
        'source_partition_key': pl.String,
        'source_partition_value': pl.String,
        'target_table': pl.String,
        'load_start_dtime': pl.Datetime,
        'load_end_dtime': pl.Datetime,
        'load_status': pl.String,
        'rows_loaded': pl.Int64,
        'error_message': pl.String,
    },
    'tech_source_partitions': {
        'source_id': pl.Int32,
        'source_name': pl.String,
        'target_table': pl.String,
        'partition_key': pl.String,
        'partition_value': pl.String,
        'partition_path': pl.String,
        'manifest_fingerprint': pl.String,
        'last_processed_status': pl.String,
        'first_load_id': pl.Int64,
        'last_load_id': pl.Int64,
        'row_create_dtime': pl.Datetime,
        'row_update_dtime': pl.Datetime,
    },
    'client_monthly_attrs_scd1': {
        'client_id': pl.String,
        'attribute_id': pl.Int32,
        'attribute_value': pl.String,
        'source_id': pl.Int32,
        'row_update_dtime': pl.Datetime,
        'row_loading_id': pl.Int64,
        'row_hash_val': pl.String,
        'report_dt': pl.String,
    },
    'client_daily_attrs_scd2': {
        'client_id': pl.String,
        'attribute_id': pl.Int32,
        'attribute_value': pl.String,
        'row_actual_from': pl.String,
        'source_id': pl.Int32,
        'row_update_dtime': pl.Datetime,
        'row_loading_id': pl.Int64,
        'row_hash_val': pl.String,
        'row_actual_to': pl.String,
    },
}

## 1. Валидация режима и путей запуска

In [ ]:
def validate_run_mode(run_mode: str) -> str:
    """
    Проверяет допустимость режима выполнения Polars-реализации.

    Режимы полностью совпадают с контрактом Spark-версии:
    - `production`: без удаления существующих артефактов;
    - `debug`: с возможностью полной очистки только выделенного debug-root каталога.

    Параметры:
        run_mode:
            Входное строковое имя режима.

    Возвращаемое значение:
        Нормализованная строка режима выполнения.

    Исключения:
        ValueError: если режим не распознан.

    Пример входа:
        validate_run_mode('production')

    Пример выхода:
        'production'
    """
    normalized_mode = run_mode.strip().lower()
    if normalized_mode not in {'production', 'debug'}:
        raise ValueError("Допустимы только режимы 'production' и 'debug'.")
    return normalized_mode

In [ ]:
print(validate_run_mode('DEBUG'))

In [ ]:
def resolve_runtime_paths(
    sources_root: str | Path | None = None,
    warehouse_root: str | Path | None = None,
    run_mode: str = 'production',
    debug_root: str | Path | None = None,
) -> dict[str, str]:
    """
    Определяет каталоги источников и path-based хранилища для Polars-реализации.

    Функция повторяет контракт Spark notebook, чтобы оба варианта запускались одинаково и могли
    использовать одни и те же тестовые фикстуры. Пути только вычисляются и валидируются; создание
    каталогов выполняется позже отдельной функцией и только в нужный момент.

    Параметры:
        sources_root:
            Необязательный путь к директории `sources`.
        warehouse_root:
            Production-root каталога хранилища.
        run_mode:
            `production` или `debug`.
        debug_root:
            Необязательный debug-root каталог.

    Возвращаемое значение:
        Словарь с путями и выбранным режимом.

    Пример входа:
        resolve_runtime_paths('/tmp/sources', '/tmp/warehouse_prod', 'debug', '/tmp/warehouse_debug')

    Пример выхода:
        {
            'sources_root': '/tmp/sources',
            'production_warehouse_root': '/tmp/warehouse_prod',
            'debug_warehouse_root': '/tmp/warehouse_debug',
            'active_warehouse_root': '/tmp/warehouse_debug',
            'run_mode': 'debug'
        }
    """
    normalized_mode = validate_run_mode(run_mode)

    source_candidates: list[Path] = []
    if sources_root is not None:
        source_candidates.append(Path(sources_root))
    if os.environ.get('DBSCORING_SOURCES_ROOT'):
        source_candidates.append(Path(os.environ['DBSCORING_SOURCES_ROOT']))
    for base_dir in (Path.cwd(), *Path.cwd().parents):
        source_candidates.extend(
            [
                base_dir / 'source' / 'sources',
                base_dir / 'data' / 'sources',
                base_dir / 'sources',
            ]
        )

    resolved_sources_root = None
    for candidate in source_candidates:
        if candidate.exists():
            resolved_sources_root = candidate.resolve()
            break
    if resolved_sources_root is None:
        raise FileNotFoundError('Не удалось определить каталог с исходными данными sources.')

    production_root = Path(
        warehouse_root
        or os.environ.get('DBSCORING_WAREHOUSE_ROOT')
        or (Path.cwd() / 'dbscoring' / 'warehouse_polars')
    ).resolve()
    debug_root_path = Path(
        debug_root
        or os.environ.get('DBSCORING_DEBUG_ROOT')
        or production_root.parent / f'{production_root.name}_debug'
    ).resolve()
    if production_root == debug_root_path:
        raise ValueError('Production-root и debug-root не должны совпадать.')

    active_root = debug_root_path if normalized_mode == 'debug' else production_root
    return {
        'sources_root': str(resolved_sources_root),
        'production_warehouse_root': str(production_root),
        'debug_warehouse_root': str(debug_root_path),
        'active_warehouse_root': str(active_root),
        'run_mode': normalized_mode,
    }

In [ ]:
POLARS_EXAMPLE_DIR = TemporaryDirectory()
POLARS_EXAMPLE_ROOT = Path(POLARS_EXAMPLE_DIR.name)
POLARS_EXAMPLE_SOURCES = POLARS_EXAMPLE_ROOT / 'sources'
for source_name, source_meta in SOURCE_REGISTRY.items():
    (POLARS_EXAMPLE_SOURCES / source_name / f"{source_meta['partition_key']}='2024-01-01'").mkdir(parents=True, exist_ok=True)
print(
    json.dumps(
        resolve_runtime_paths(
            sources_root=POLARS_EXAMPLE_SOURCES,
            warehouse_root=POLARS_EXAMPLE_ROOT / 'warehouse_prod',
            run_mode='debug',
            debug_root=POLARS_EXAMPLE_ROOT / 'warehouse_debug',
        ),
        ensure_ascii=False,
        indent=2,
    )
)

## 2. Поиск и контроль партиций

In [ ]:
def parse_partition_directory_name(partition_dir_name: str) -> tuple[str, str]:
    """
    Разбирает имя директории партиции формата `key='value'`.

    Функция используется одинаково и для daily-, и для monthly-источников. Ошибка в имени каталога
    должна быть обнаружена до фактического чтения parquet-файлов, поэтому проверка выполняется строго.

    Параметры:
        partition_dir_name:
            Строковое имя директории партиции.

    Возвращаемое значение:
        Кортеж `(partition_key, partition_value)`.

    Исключения:
        ValueError: если формат имени некорректен.

    Пример входа:
        parse_partition_directory_name("row_actual_to='2024-03-31'")

    Пример выхода:
        ('row_actual_to', '2024-03-31')
    """
    match = PARTITION_NAME_PATTERN.match(partition_dir_name)
    if match is None:
        raise ValueError(f"Имя каталога '{partition_dir_name}' не соответствует шаблону key='value'.")
    return match.group('key'), match.group('value')

In [ ]:
print(parse_partition_directory_name("row_actual_to='2024-03-31'"))

In [ ]:
def discover_source_partitions(
    sources_root: str | Path,
    source_registry: dict[str, dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    """
    Находит и описывает все партиции входных источников без чтения parquet-файлов.

    Функция возвращает минимально достаточный набор полей для оркестратора: источник, ключ партиции,
    значение партиции, путь к директории, target table и тип периодичности обновления.

    Параметры:
        sources_root:
            Путь к каталогу `sources`.
        source_registry:
            Необязательный реестр источников.

    Возвращаемое значение:
        Отсортированный список словарей по всем найденным партициям.

    Исключения:
        FileNotFoundError: если отсутствует директория источника.
        ValueError: если ключ партиции не совпал с ожидаемым.

    Пример входа:
        discover_source_partitions('/tmp/sources')

    Пример выхода:
        [{'source_name': 'deb_cards_info', 'partition_key': 'report_dt', ...}]
    """
    registry = source_registry or SOURCE_REGISTRY
    result: list[dict[str, Any]] = []
    sources_root_path = Path(sources_root)

    for source_name, source_meta in registry.items():
        source_dir = sources_root_path / source_name
        if not source_dir.exists():
            raise FileNotFoundError(f'Отсутствует директория источника: {source_dir}')
        for partition_dir in sorted(path for path in source_dir.iterdir() if path.is_dir() and not path.name.startswith('.')):
            partition_key, partition_value = parse_partition_directory_name(partition_dir.name)
            if partition_key != source_meta['partition_key']:
                raise ValueError(
                    f"Для источника {source_name} ожидался ключ {source_meta['partition_key']}, но найден {partition_key}."
                )
            result.append(
                {
                    'source_name': source_name,
                    'source_id': source_meta['source_id'],
                    'partition_key': partition_key,
                    'partition_value': partition_value,
                    'partition_path': str(partition_dir.resolve()),
                    'target_table': source_meta['target_table'],
                    'update_frequency': source_meta['update_frequency'],
                }
            )
    return sorted(result, key=lambda row: (row['source_id'], row['partition_value'], row['partition_path']))

In [ ]:
print(discover_source_partitions(POLARS_EXAMPLE_SOURCES)[0])

In [ ]:
def build_manifest_fingerprint(partition_path: str | Path) -> str:
    """
    Вычисляет fingerprint партиции по именам, размерам и времени модификации parquet-файлов.

    Эта операция дёшево стоит по I/O и позволяет быстро отличать новую партицию от уже обработанной,
    не читая её содержимое. В fingerprint включаются только файлы `*.parquet`.

    Параметры:
        partition_path:
            Путь к директории одной партиции.

    Возвращаемое значение:
        SHA-256 fingerprint в текстовом виде.

    Исключения:
        FileNotFoundError: если parquet-файлы не найдены.

    Пример входа:
        build_manifest_fingerprint('/tmp/sources/credit_cards_info/report_dt='2024-01-31'')

    Пример выхода:
        'a18d96f0...'
    """
    partition_dir = Path(partition_path)
    parquet_files = sorted(partition_dir.glob('*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(f'В директории {partition_dir} не найдено parquet-файлов.')
    manifest_rows = []
    for parquet_file in parquet_files:
        stats = parquet_file.stat()
        manifest_rows.append({'name': parquet_file.name, 'size': stats.st_size, 'mtime_ns': stats.st_mtime_ns})
    payload = json.dumps(manifest_rows, ensure_ascii=False, sort_keys=True).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

In [ ]:
POLARS_MANIFEST_DIR = POLARS_EXAMPLE_ROOT / 'manifest_example'
POLARS_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
(POLARS_MANIFEST_DIR / 'part-00000.parquet').write_bytes(b'polars-a')
(POLARS_MANIFEST_DIR / 'part-00001.parquet').write_bytes(b'polars-b')
print(build_manifest_fingerprint(POLARS_MANIFEST_DIR))

## 3. Подготовка хранилища и справочников

In [ ]:
def infer_attribute_data_type(attribute_name: str) -> str:
    """
    Определяет логический тип атрибута по его имени.

    Эта функция используется только для наполнения `dim_attributes` и полностью повторяет логику Spark notebook,
    чтобы оба контура строили одинаковый справочник атрибутов.

    Параметры:
        attribute_name:
            Имя атрибута источника.

    Возвращаемое значение:
        Одно из значений `DATE`, `DECIMAL`, `INT`, `STRING`.

    Пример входа:
        infer_attribute_data_type('card_active_1m_nflag')

    Пример выхода:
        'INT'
    """
    if attribute_name.endswith(('_dt', '_from', '_to')):
        return 'DATE'
    if attribute_name.endswith('_amt'):
        return 'DECIMAL'
    if attribute_name.endswith(('_nflag', '_nfalg', '_qty')):
        return 'INT'
    return 'STRING'

In [ ]:
print(infer_attribute_data_type('card_active_1m_nflag'))

In [ ]:
def initialize_warehouse_layout(warehouse_root: str | Path) -> dict[str, str]:
    """
    Создаёт физическую структуру path-based хранилища для Polars и инициализирует маленькие parquet-таблицы.

    Для небольших таблиц (`dim_*`, `load_log`, `tech_source_partitions`) создаются пустые parquet-файлы с жёстко
    заданной схемой. Для больших факт-таблиц создаются только каталоги: данные будут появляться в них по мере
    поступления новых партиций источников.

    Параметры:
        warehouse_root:
            Корневой каталог хранилища.

    Возвращаемое значение:
        Словарь `{table_name: table_location}`.

    Пример входа:
        initialize_warehouse_layout('/tmp/warehouse_debug')

    Пример выхода:
        {'dim_sources': '/tmp/warehouse_debug/dim_sources', ...}
    """
    warehouse_root_path = Path(warehouse_root).resolve()
    warehouse_root_path.mkdir(parents=True, exist_ok=True)
    locations = {}
    for table_name, schema in POLARS_TABLE_SCHEMAS.items():
        table_dir = warehouse_root_path / table_name
        table_dir.mkdir(parents=True, exist_ok=True)
        locations[table_name] = str(table_dir)
        if table_name in {'client_monthly_attrs_scd1', 'client_daily_attrs_scd2'}:
            continue
        data_path = table_dir / 'data.parquet'
        if not data_path.exists():
            pl.DataFrame(schema=schema).write_parquet(data_path)
    return locations

In [ ]:
polars_example_paths = initialize_warehouse_layout(POLARS_EXAMPLE_ROOT / 'warehouse_debug')
print(sorted(polars_example_paths))

In [ ]:
def upsert_reference_dimensions(warehouse_root: str | Path, load_timestamp: dt.datetime) -> dict[str, int]:
    """
    Перезаписывает маленькие справочники `dim_sources` и `dim_attributes` на основе текущего `SOURCE_REGISTRY`.

    Операция полностью локальна и безопасна по стоимости, потому что справочники малы. Это позволяет
    гарантировать их согласованность с реестром источников на каждом запуске без сложных частичных merge-операций.

    Параметры:
        warehouse_root:
            Корневой каталог path-based хранилища.
        load_timestamp:
            Техническая временная метка запуска.

    Возвращаемое значение:
        Словарь с количеством записанных строк по двум справочникам.

    Пример входа:
        upsert_reference_dimensions('/tmp/warehouse_debug', dt.datetime(2024, 1, 31, 12, 0, 0))

    Пример выхода:
        {'dim_sources_rows': 3, 'dim_attributes_rows': 24}
    """
    warehouse_root_path = Path(warehouse_root)
    dim_sources_rows = []
    for source_name, source_meta in SOURCE_REGISTRY.items():
        dim_sources_rows.append(
            {
                'source_id': source_meta['source_id'],
                'source_name': source_name,
                'source_description': source_meta['source_description'],
                'update_frequency': source_meta['update_frequency'],
                'row_create_dtime': load_timestamp,
                'row_update_dtime': load_timestamp,
                'valid_from': load_timestamp,
                'valid_to': FUTURE_DTTM,
            }
        )

    dim_attributes_rows = []
    for source_name, source_meta in SOURCE_REGISTRY.items():
        for attribute_name in source_meta['columns']:
            dim_attributes_rows.append(
                {
                    'attribute_id': ATTRIBUTE_ID_BY_NAME[attribute_name],
                    'attribute_name': attribute_name,
                    'attribute_description': f'Атрибут {attribute_name} из источника {source_name}.',
                    'data_type': infer_attribute_data_type(attribute_name),
                    'source_id': source_meta['source_id'],
                    'update_frequency': source_meta['update_frequency'],
                    'row_create_dtime': load_timestamp,
                    'row_update_dtime': load_timestamp,
                }
            )

    pl.DataFrame(dim_sources_rows, schema=POLARS_TABLE_SCHEMAS['dim_sources']).write_parquet(
        warehouse_root_path / 'dim_sources' / 'data.parquet'
    )
    pl.DataFrame(dim_attributes_rows, schema=POLARS_TABLE_SCHEMAS['dim_attributes']).write_parquet(
        warehouse_root_path / 'dim_attributes' / 'data.parquet'
    )
    return {'dim_sources_rows': len(dim_sources_rows), 'dim_attributes_rows': len(dim_attributes_rows)}

In [ ]:
print(upsert_reference_dimensions(POLARS_EXAMPLE_ROOT / 'warehouse_debug', dt.datetime(2024, 1, 31, 12, 0, 0)))

In [ ]:
def load_partition_state_map(warehouse_root: str | Path) -> dict[tuple[str, str, str], dict[str, Any]]:
    """
    Загружает таблицу `tech_source_partitions` в Python-словарь для быстрых решений `new / skip / fail`.

    Поскольку техническая таблица мала, её удобно целиком держать в памяти и не читать по одной строке
    при обработке каждой партиции.

    Параметры:
        warehouse_root:
            Корневой каталог хранилища.

    Возвращаемое значение:
        Словарь с ключом `(source_name, partition_key, partition_value)`.

    Пример входа:
        load_partition_state_map('/tmp/warehouse_debug')

    Пример выхода:
        {('credit_cards_info', 'report_dt', '2024-01-31'): {...}}
    """
    table_path = Path(warehouse_root) / 'tech_source_partitions' / 'data.parquet'
    frame = pl.read_parquet(table_path)
    state_map: dict[tuple[str, str, str], dict[str, Any]] = {}
    for row in frame.to_dicts():
        state_map[(row['source_name'], row['partition_key'], row['partition_value'])] = row
    return state_map

In [ ]:
print(load_partition_state_map(POLARS_EXAMPLE_ROOT / 'warehouse_debug'))

In [ ]:
def determine_partition_action(existing_state: dict[str, Any] | None, current_fingerprint: str) -> str:
    """
    Определяет действие для партиции по правилам идемпотентности.

    Если партиция ещё не встречалась, она должна быть загружена. Если fingerprint совпадает с уже известным,
    партицию можно пропустить без повторного чтения. Если fingerprint изменился, это ошибка upstream-данных,
    и загрузка должна завершиться аварийно.

    Параметры:
        existing_state:
            Предыдущее состояние партиции или `None`.
        current_fingerprint:
            Новый fingerprint текущей директории партиции.

    Возвращаемое значение:
        `new`, `skip` или `fail`.

    Пример входа:
        determine_partition_action({'manifest_fingerprint': 'abc'}, 'abc')

    Пример выхода:
        'skip'
    """
    if existing_state is None:
        return 'new'
    if existing_state['manifest_fingerprint'] == current_fingerprint:
        return 'skip'
    return 'fail'

In [ ]:
print(determine_partition_action(None, 'abc'))
print(determine_partition_action({'manifest_fingerprint': 'abc'}, 'abc'))
print(determine_partition_action({'manifest_fingerprint': 'abc'}, 'xyz'))

## 4. Локальная запись фактов и технических таблиц

In [ ]:
def append_partition_to_fact_table(partition_record: dict[str, Any], load_id: int, warehouse_root: str | Path) -> int:
    """
    Читает одну новую партицию источника, преобразует её в EAV и append-only записывает в целевой parquet-каталог.

    Для учебной Polars-версии используется явное чтение parquet-файлов и локальная запись результата в каталог
    конкретной partition-value. Контракт полностью повторяет Spark-версию: уже обработанные партиции сюда не
    попадают, а значит запись выполняется только в новый partition directory.

    Параметры:
        partition_record:
            Словарь из `discover_source_partitions`.
        load_id:
            Идентификатор текущего загрузочного события.
        warehouse_root:
            Корневой каталог path-based хранилища.

    Возвращаемое значение:
        Количество записанных строк.

    Пример входа:
        append_partition_to_fact_table(partition_record, load_id=101, warehouse_root='/tmp/warehouse_debug')

    Пример выхода:
        22
    """
    source_name = partition_record['source_name']
    source_meta = SOURCE_REGISTRY[source_name]
    partition_path = Path(partition_record['partition_path'])
    raw_frame = pl.read_parquet(str(partition_path / '*.parquet'))
    attribute_lookup = pl.DataFrame(
        {
            'attribute_name': list(source_meta['columns']),
            'attribute_id': [ATTRIBUTE_ID_BY_NAME[name] for name in source_meta['columns']],
        }
    )

    id_columns = ['id', 'row_update_dtime', 'loading_id', 'row_hash_val']
    if source_meta['update_frequency'] == 'daily':
        id_columns.append('row_actual_from')

    melted = raw_frame.unpivot(
        index=id_columns,
        on=list(source_meta['columns']),
        variable_name='attribute_name',
        value_name='attribute_value',
    )
    enriched = melted.join(attribute_lookup, on='attribute_name', how='inner')

    if source_meta['update_frequency'] == 'monthly':
        result_frame = enriched.select(
            pl.col('id').cast(pl.String).alias('client_id'),
            pl.col('attribute_id').cast(pl.Int32),
            pl.col('attribute_value').cast(pl.String),
            pl.lit(source_meta['source_id']).cast(pl.Int32).alias('source_id'),
            pl.col('row_update_dtime').cast(pl.Datetime),
            pl.col('loading_id').cast(pl.Int64).alias('row_loading_id'),
            pl.col('row_hash_val').cast(pl.String),
            pl.lit(partition_record['partition_value']).cast(pl.String).alias('report_dt'),
        )
        target_partition_dir = (
            Path(warehouse_root)
            / source_meta['target_table']
            / f"report_dt='{partition_record['partition_value']}'"
        )
    else:
        result_frame = enriched.select(
            pl.col('id').cast(pl.String).alias('client_id'),
            pl.col('attribute_id').cast(pl.Int32),
            pl.col('attribute_value').cast(pl.String),
            pl.col('row_actual_from').cast(pl.String),
            pl.lit(source_meta['source_id']).cast(pl.Int32).alias('source_id'),
            pl.col('row_update_dtime').cast(pl.Datetime),
            pl.col('loading_id').cast(pl.Int64).alias('row_loading_id'),
            pl.col('row_hash_val').cast(pl.String),
            pl.lit(partition_record['partition_value']).cast(pl.String).alias('row_actual_to'),
        )
        target_partition_dir = (
            Path(warehouse_root)
            / source_meta['target_table']
            / f"row_actual_to='{partition_record['partition_value']}'"
        )

    target_partition_dir.mkdir(parents=True, exist_ok=True)
    result_frame.write_parquet(target_partition_dir / f'part-{load_id:05d}.parquet')
    return result_frame.height

In [ ]:
polars_monthly_partition_dir = POLARS_EXAMPLE_SOURCES / 'credit_cards_info' / "report_dt='2024-01-31'"
polars_monthly_partition_dir.mkdir(parents=True, exist_ok=True)
pl.DataFrame(
    [
        {
            'id': 'C001',
            'client_income_amt': 100000.0,
            'oi_total_amt': 150000.0,
            'act_pl_os_rub_amt': 75000.0,
            'payroll_client_nflag': 1,
            'inf_payroll_rub_amt': 50000.0,
            'legal_entity_amt': 25000.0,
            'inc_avg_risk_rub_amt': 1500.0,
            'otf_loan_rub_amt': 5000.0,
            'otf_fee_rub_amt': 250.0,
            'inf_transfer_rub_amt': 3000.0,
            'cc_ever_nflag': 1,
            'row_update_dtime': dt.datetime(2024, 1, 31, 10, 0, 0),
            'loading_id': 101,
            'row_hash_val': 'hash-credit-20240131-C001',
        }
    ]
).write_parquet(polars_monthly_partition_dir / 'part-00000.parquet')
polars_partition_record = next(
    item
    for item in discover_source_partitions(POLARS_EXAMPLE_SOURCES)
    if item['source_name'] == 'credit_cards_info' and item['partition_value'] == '2024-01-31'
)
print(append_partition_to_fact_table(polars_partition_record, 101, POLARS_EXAMPLE_ROOT / 'warehouse_debug'))

In [ ]:
def append_load_log_entry(warehouse_root: str | Path, load_log_row: dict[str, Any]) -> None:
    """
    Добавляет одну строку в локальный parquet-журнал `load_log`.

    Функция читает текущий журнал целиком, добавляет одну новую строку и перезаписывает только маленький
    файл журнала. Такая операция допустима, потому что `load_log` по контракту остаётся компактной таблицей.

    Параметры:
        warehouse_root:
            Корневой каталог хранилища.
        load_log_row:
            Одна запись журнала в виде словаря.

    Возвращаемое значение:
        `None`.

    Пример входа:
        append_load_log_entry('/tmp/warehouse_debug', {...})

    Пример выхода:
        В `load_log` появляется новая строка.
    """
    data_path = Path(warehouse_root) / 'load_log' / 'data.parquet'
    existing_frame = pl.read_parquet(data_path)
    new_frame = pl.DataFrame([load_log_row], schema=POLARS_TABLE_SCHEMAS['load_log'])
    pl.concat([existing_frame, new_frame], how='vertical_relaxed').write_parquet(data_path)

In [ ]:
append_load_log_entry(
    POLARS_EXAMPLE_ROOT / 'warehouse_debug',
    {
        'load_id': 101,
        'source_id': 2,
        'source_name': 'credit_cards_info',
        'source_partition_key': 'report_dt',
        'source_partition_value': '2024-01-31',
        'target_table': 'client_monthly_attrs_scd1',
        'load_start_dtime': dt.datetime(2024, 1, 31, 10, 0, 0),
        'load_end_dtime': dt.datetime(2024, 1, 31, 10, 0, 1),
        'load_status': 'loaded',
        'rows_loaded': 11,
        'error_message': None,
    },
)
print(pl.read_parquet(POLARS_EXAMPLE_ROOT / 'warehouse_debug' / 'load_log' / 'data.parquet').tail(1))

In [ ]:
def merge_partition_state_entry(warehouse_root: str | Path, state_row: dict[str, Any]) -> None:
    """
    Выполняет локальный upsert записи в parquet-таблицу `tech_source_partitions`.

    Алгоритм полностью повторяет смысл Spark-версии: берётся маленькая техническая таблица, к ней добавляется
    новая строка состояния, после чего остаётся только последняя версия на комбинацию
    `(source_name, partition_key, partition_value)`.

    Параметры:
        warehouse_root:
            Корневой каталог хранилища.
        state_row:
            Одна запись состояния партиции.

    Возвращаемое значение:
        `None`.

    Пример входа:
        merge_partition_state_entry('/tmp/warehouse_debug', {...})

    Пример выхода:
        В parquet-таблице сохраняется одна актуальная строка на одну партицию.
    """
    data_path = Path(warehouse_root) / 'tech_source_partitions' / 'data.parquet'
    existing_frame = pl.read_parquet(data_path)
    new_frame = pl.DataFrame([state_row], schema=POLARS_TABLE_SCHEMAS['tech_source_partitions'])
    merged_frame = pl.concat([existing_frame, new_frame], how='vertical_relaxed').sort(
        by=['source_name', 'partition_key', 'partition_value', 'row_update_dtime', 'last_load_id'],
        descending=[False, False, False, True, True],
    )
    unique_frame = merged_frame.unique(
        subset=['source_name', 'partition_key', 'partition_value'],
        keep='first',
        maintain_order=True,
    )
    unique_frame.write_parquet(data_path)

In [ ]:
merge_partition_state_entry(
    POLARS_EXAMPLE_ROOT / 'warehouse_debug',
    {
        'source_id': 2,
        'source_name': 'credit_cards_info',
        'target_table': 'client_monthly_attrs_scd1',
        'partition_key': 'report_dt',
        'partition_value': '2024-01-31',
        'partition_path': str(polars_monthly_partition_dir),
        'manifest_fingerprint': build_manifest_fingerprint(polars_monthly_partition_dir),
        'last_processed_status': 'loaded',
        'first_load_id': 101,
        'last_load_id': 101,
        'row_create_dtime': dt.datetime(2024, 1, 31, 10, 0, 0),
        'row_update_dtime': dt.datetime(2024, 1, 31, 10, 0, 1),
    },
)
print(load_partition_state_map(POLARS_EXAMPLE_ROOT / 'warehouse_debug'))

In [ ]:
def clear_debug_artifacts(target_root: str | Path, approved_debug_root: str | Path) -> str:
    """
    Полностью удаляет debug-root каталог только после строгой проверки безопасности.

    Политика совпадает со Spark-версией: разрешается удалять только заранее подтверждённый каталог,
    который явно помечен как debug.

    Параметры:
        target_root:
            Каталог, который нужно очистить.
        approved_debug_root:
            Заранее разрешённый debug-root каталог.

    Возвращаемое значение:
        Нормализованный путь к удалённому каталогу.

    Исключения:
        ValueError: если путь небезопасен.

    Пример входа:
        clear_debug_artifacts('/tmp/warehouse_debug', '/tmp/warehouse_debug')

    Пример выхода:
        '/tmp/warehouse_debug'
    """
    target_path = Path(target_root).resolve()
    approved_path = Path(approved_debug_root).resolve()
    if target_path != approved_path:
        raise ValueError('Разрешено удалять только заранее подтверждённый debug-root каталог.')
    if 'debug' not in target_path.name.lower():
        raise ValueError('Для удаления каталог должен явно содержать маркер debug в имени.')
    if str(target_path) in {'/', str(Path.home().resolve())}:
        raise ValueError('Удаление системных корней запрещено.')
    if target_path.exists():
        shutil.rmtree(target_path)
    return str(target_path)

In [ ]:
polars_cleanup_root = POLARS_EXAMPLE_ROOT / 'safe_debug_directory'
polars_cleanup_root.mkdir(parents=True, exist_ok=True)
(polars_cleanup_root / 'temporary.txt').write_text('debug payload', encoding='utf-8')
print(clear_debug_artifacts(polars_cleanup_root, polars_cleanup_root))
print(polars_cleanup_root.exists())

In [ ]:
def run_warehouse_update(
    sources_root: str | Path | None = None,
    warehouse_root: str | Path | None = None,
    run_mode: str = 'production',
    debug_root: str | Path | None = None,
    cleanup_existing_debug_root: bool = False,
) -> dict[str, Any]:
    """
    Выполняет полный локальный инкрементальный прогон хранилища данных на Polars.

    Алгоритм эквивалентен Spark-версии:
    1. Разрешить и провалидировать пути.
    2. При необходимости очистить debug-root.
    3. Инициализировать layout хранилища.
    4. Пересобрать маленькие справочники.
    5. Для каждой партиции принять решение `new / skip / fail` по fingerprint.
    6. Для `new` записать данные в новую partition directory, для `skip` и `fail` — обновить только аудит.

    Параметры:
        sources_root:
            Необязательный путь к каталогу `sources`.
        warehouse_root:
            Production-root каталога хранилища.
        run_mode:
            `production` или `debug`.
        debug_root:
            Необязательный debug-root.
        cleanup_existing_debug_root:
            Флаг очистки debug-root перед запуском.

    Возвращаемое значение:
        Сводный словарь по результату запуска.

    Пример входа:
        run_warehouse_update('/tmp/sources', '/tmp/warehouse_prod', 'debug', '/tmp/warehouse_debug', True)

    Пример выхода:
        {'loaded_partitions': 3, 'skipped_partitions': 0, 'failed_partitions': 0, ...}
    """
    runtime_paths = resolve_runtime_paths(
        sources_root=sources_root,
        warehouse_root=warehouse_root,
        run_mode=run_mode,
        debug_root=debug_root,
    )
    active_warehouse_root = Path(runtime_paths['active_warehouse_root'])
    if runtime_paths['run_mode'] == 'debug' and cleanup_existing_debug_root:
        clear_debug_artifacts(active_warehouse_root, runtime_paths['debug_warehouse_root'])

    table_locations = initialize_warehouse_layout(active_warehouse_root)
    run_timestamp = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
    upsert_reference_dimensions(active_warehouse_root, run_timestamp)

    state_map = load_partition_state_map(active_warehouse_root)
    discovered_partitions = discover_source_partitions(runtime_paths['sources_root'])
    load_log_path = active_warehouse_root / 'load_log' / 'data.parquet'
    current_load_id = int(pl.read_parquet(load_log_path)['load_id'].max() or 0)

    loaded_partitions = 0
    skipped_partitions = 0
    failed_partitions = 0
    processed_partitions = []

    for partition_record in discovered_partitions:
        current_load_id += 1
        load_start = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
        state_key = (
            partition_record['source_name'],
            partition_record['partition_key'],
            partition_record['partition_value'],
        )
        current_fingerprint = build_manifest_fingerprint(partition_record['partition_path'])
        existing_state = state_map.get(state_key)
        action = determine_partition_action(existing_state, current_fingerprint)

        if action == 'skip':
            skipped_partitions += 1
            append_load_log_entry(
                active_warehouse_root,
                {
                    'load_id': current_load_id,
                    'source_id': partition_record['source_id'],
                    'source_name': partition_record['source_name'],
                    'source_partition_key': partition_record['partition_key'],
                    'source_partition_value': partition_record['partition_value'],
                    'target_table': partition_record['target_table'],
                    'load_start_dtime': load_start,
                    'load_end_dtime': dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0),
                    'load_status': 'skipped',
                    'rows_loaded': 0,
                    'error_message': None,
                },
            )
            updated_state = dict(existing_state)
            updated_state['last_processed_status'] = 'skipped'
            updated_state['last_load_id'] = current_load_id
            updated_state['row_update_dtime'] = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
            merge_partition_state_entry(active_warehouse_root, updated_state)
            state_map[state_key] = updated_state
            processed_partitions.append(
                {
                    'source_name': partition_record['source_name'],
                    'partition_value': partition_record['partition_value'],
                    'action': 'skipped',
                    'rows_loaded': 0,
                }
            )
            continue

        if action == 'fail':
            failed_partitions += 1
            error_message = (
                'Обнаружено изменение уже обработанной партиции: '
                f"{partition_record['source_name']} / {partition_record['partition_key']}={partition_record['partition_value']}"
            )
            append_load_log_entry(
                active_warehouse_root,
                {
                    'load_id': current_load_id,
                    'source_id': partition_record['source_id'],
                    'source_name': partition_record['source_name'],
                    'source_partition_key': partition_record['partition_key'],
                    'source_partition_value': partition_record['partition_value'],
                    'target_table': partition_record['target_table'],
                    'load_start_dtime': load_start,
                    'load_end_dtime': dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0),
                    'load_status': 'failed',
                    'rows_loaded': 0,
                    'error_message': error_message,
                },
            )
            raise RuntimeError(error_message)

        try:
            rows_loaded = append_partition_to_fact_table(partition_record, current_load_id, active_warehouse_root)
            loaded_partitions += 1
            load_end = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
            append_load_log_entry(
                active_warehouse_root,
                {
                    'load_id': current_load_id,
                    'source_id': partition_record['source_id'],
                    'source_name': partition_record['source_name'],
                    'source_partition_key': partition_record['partition_key'],
                    'source_partition_value': partition_record['partition_value'],
                    'target_table': partition_record['target_table'],
                    'load_start_dtime': load_start,
                    'load_end_dtime': load_end,
                    'load_status': 'loaded',
                    'rows_loaded': rows_loaded,
                    'error_message': None,
                },
            )
            new_state = {
                'source_id': partition_record['source_id'],
                'source_name': partition_record['source_name'],
                'target_table': partition_record['target_table'],
                'partition_key': partition_record['partition_key'],
                'partition_value': partition_record['partition_value'],
                'partition_path': partition_record['partition_path'],
                'manifest_fingerprint': current_fingerprint,
                'last_processed_status': 'loaded',
                'first_load_id': existing_state['first_load_id'] if existing_state else current_load_id,
                'last_load_id': current_load_id,
                'row_create_dtime': existing_state['row_create_dtime'] if existing_state else load_start,
                'row_update_dtime': load_end,
            }
            merge_partition_state_entry(active_warehouse_root, new_state)
            state_map[state_key] = new_state
            processed_partitions.append(
                {
                    'source_name': partition_record['source_name'],
                    'partition_value': partition_record['partition_value'],
                    'action': 'loaded',
                    'rows_loaded': rows_loaded,
                }
            )
        except Exception as exc:
            failed_partitions += 1
            append_load_log_entry(
                active_warehouse_root,
                {
                    'load_id': current_load_id,
                    'source_id': partition_record['source_id'],
                    'source_name': partition_record['source_name'],
                    'source_partition_key': partition_record['partition_key'],
                    'source_partition_value': partition_record['partition_value'],
                    'target_table': partition_record['target_table'],
                    'load_start_dtime': load_start,
                    'load_end_dtime': dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0),
                    'load_status': 'failed',
                    'rows_loaded': 0,
                    'error_message': str(exc),
                },
            )
            raise

    return {
        'run_mode': runtime_paths['run_mode'],
        'sources_root': runtime_paths['sources_root'],
        'active_warehouse_root': str(active_warehouse_root),
        'loaded_partitions': loaded_partitions,
        'skipped_partitions': skipped_partitions,
        'failed_partitions': failed_partitions,
        'processed_partitions': processed_partitions,
        'table_locations': table_locations,
    }

In [ ]:
POLARS_FULL_ROOT = POLARS_EXAMPLE_ROOT / 'full_pipeline'
POLARS_FULL_SOURCES = POLARS_FULL_ROOT / 'sources'
POLARS_FULL_DEBUG = POLARS_FULL_ROOT / 'warehouse_debug'
for source_name in SOURCE_REGISTRY:
    (POLARS_FULL_SOURCES / source_name).mkdir(parents=True, exist_ok=True)
(POLARS_FULL_SOURCES / 'credit_cards_info' / "report_dt='2024-03-31'").mkdir(parents=True, exist_ok=True)
(POLARS_FULL_SOURCES / 'deb_cards_info' / "report_dt='2024-03-31'").mkdir(parents=True, exist_ok=True)
(POLARS_FULL_SOURCES / 'client_cards_daily' / "row_actual_to='2024-03-31'").mkdir(parents=True, exist_ok=True)

pl.DataFrame(
    [
        {
            'id': 'C100',
            'client_income_amt': 120000.0,
            'oi_total_amt': 190000.0,
            'act_pl_os_rub_amt': 80000.0,
            'payroll_client_nflag': 1,
            'inf_payroll_rub_amt': 62000.0,
            'legal_entity_amt': 24000.0,
            'inc_avg_risk_rub_amt': 900.0,
            'otf_loan_rub_amt': 4000.0,
            'otf_fee_rub_amt': 150.0,
            'inf_transfer_rub_amt': 1000.0,
            'cc_ever_nflag': 1,
            'row_update_dtime': dt.datetime(2024, 3, 31, 9, 0, 0),
            'loading_id': 201,
            'row_hash_val': 'credit-20240331-C100',
        }
    ]
).write_parquet(POLARS_FULL_SOURCES / 'credit_cards_info' / "report_dt='2024-03-31'" / 'part-00000.parquet')
pl.DataFrame(
    [
        {
            'id': 'C100',
            'onl_bank_active_1m_nfalg': 1,
            'auto_pay_active_qty': 2,
            'cl_income_1m_amt': 70000.0,
            'dep_acc_1st_open_dt': '2021-06-01',
            'wdr_cash_6m_amt': 12000.0,
            'cash_op_6m_amt': 14000.0,
            'cash_3m_qty': 4,
            'lst_balance_amt': 30000.0,
            'card_active_1m_nflag': 1,
            'row_update_dtime': dt.datetime(2024, 3, 31, 9, 5, 0),
            'loading_id': 202,
            'row_hash_val': 'debit-20240331-C100',
        }
    ]
).write_parquet(POLARS_FULL_SOURCES / 'deb_cards_info' / "report_dt='2024-03-31'" / 'part-00000.parquet')
pl.DataFrame(
    [
        {
            'id': 'C100',
            'srv_mb_nflag': 1,
            'cc_stoplist': 0,
            'lne_tot_debt_int_ovrd_rub_amt': 0.0,
            'lne_tot_debt_ovrd_rub_amt': 100.0,
            'row_actual_from': '2024-03-01',
            'row_update_dtime': dt.datetime(2024, 3, 31, 9, 10, 0),
            'loading_id': 203,
            'row_hash_val': 'daily-20240331-C100',
        }
    ]
).write_parquet(POLARS_FULL_SOURCES / 'client_cards_daily' / "row_actual_to='2024-03-31'" / 'part-00000.parquet')

print(
    json.dumps(
        run_warehouse_update(
            sources_root=POLARS_FULL_SOURCES,
            warehouse_root=POLARS_FULL_ROOT / 'warehouse_prod',
            run_mode='debug',
            debug_root=POLARS_FULL_DEBUG,
            cleanup_existing_debug_root=True,
        ),
        ensure_ascii=False,
        indent=2,
    )
)

## 5. Финальный пример локального запуска

In [ ]:
if os.environ.get('DBSCORING_SKIP_FINAL_RUN', '0') == '1':
    print('Финальный прогон пропущен по флагу DBSCORING_SKIP_FINAL_RUN=1')
else:
    final_summary = run_warehouse_update(
        sources_root=os.environ.get('DBSCORING_SOURCES_ROOT'),
        warehouse_root=os.environ.get('DBSCORING_WAREHOUSE_ROOT'),
        run_mode=os.environ.get('DBSCORING_RUN_MODE', 'debug'),
        debug_root=os.environ.get('DBSCORING_DEBUG_ROOT'),
        cleanup_existing_debug_root=os.environ.get('DBSCORING_CLEAN_DEBUG', '1') == '1',
    )
    print(json.dumps(final_summary, ensure_ascii=False, indent=2))